In [0]:
# Create struct schema for products csv file

from pyspark.sql.types import *

products_schema = StructType([
    StructField("product_id", IntegerType(), True),
    StructField("category_id", IntegerType(), True),
    StructField("supplier_id", IntegerType(), True),
    StructField("price", IntegerType(), True)
])

In [0]:
# autoload csv into dataframe with schema location defined

df = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("header", "true") \
    .option(
        "cloudFiles.schemaLocation", 
        "/Volumes/first_data_engineering_project/pipeline_metadata/autoloader_metadata/schemas/bronze/bronze_products/"
        ) \
    .schema(products_schema) \
    .load(".load("/Volumes/first_data_engineering_project/landing/retail_files/products/)

In [0]:
# add bronze layer metadata columns to the dataFrame for ingestion and lineage tracking

from pyspark.sql import functions as F

bronze_products = df \
    .withColumn("source_file", F.col("_metadata.file_name")) \
    .withColumn("ingestion_timestamp", F.current_timestamp()) \
    .withColumn("file_modified_time", F.col("_metadata.file_modification_time"))


In [0]:
# create table with checkpoint location and add trigger

bronze_products.writeStream \
    .option(
        "checkpointLocation", 
        "/Volumes/first_data_engineering_project/pipeline_metadata/autoloader_metadata/checkpoints/bronze/bronze_products/"
        ) \
    .trigger(availableNow=True) \
    .toTable("first_data_engineering_project.bronze.bronze_products")